# Qwen3.5-0.8B × TinyCeNN Memory Fusion — Sequential Acceptance

This notebook ports the TinyCeNN Memory Fusion sequential experiment to `Qwen/Qwen3.5-0.8B`. Qwen3.5 already uses a 3:1 hybrid layout, so this experiment leaves the 18 native Gated DeltaNet linear-attention layers unchanged and sequentially tries to replace only the six full-attention anchors: **3, 7, 11, 15, 19, 23**.

Each replacement must pass NMSE, cosine, incremental ΔNLL and cumulative ΔNLL before the next anchor is touched. Failed rounds are saved and resumable. The notebook uses the text-only `Qwen3_5ForCausalLM` loader, so the vision tower is not loaded for this experiment.


In [ ]:
import os, sys, subprocess
from pathlib import Path
assert subprocess.run(['nvidia-smi'], check=False).returncode == 0, 'Enable a GPU runtime in Colab.'
REPO = Path('/content/TinyCeNN-LM')
if REPO.exists(): subprocess.run(['rm','-rf',str(REPO)], check=True)
subprocess.run(['git','clone','--depth','1','https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO)], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-U','git+https://github.com/huggingface/transformers.git@main','datasets','huggingface_hub','accelerate','safetensors','pytest'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO),'--no-deps'], check=True)
for p in (str(REPO), str(REPO/'src')):
    if p not in sys.path: sys.path.insert(0,p)
print('Repository:', REPO)
subprocess.run(['git','-C',str(REPO),'rev-parse','HEAD'], check=True)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_DIR = Path('/content/drive/MyDrive/TinyCeNN-LM/qwen35-0.8b-memory-fusion-sequential-r64')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None
from huggingface_hub import login, notebook_login
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('✅ Hugging Face authenticated from Colab Secret HF_TOKEN')
else:
    print('HF_TOKEN not found; opening secure Hugging Face login.')
    notebook_login()
print('Persistent output:', OUTPUT_DIR)


In [ ]:
from huggingface_hub import HfApi
from transformers import AutoConfig
BASE_MODEL = 'Qwen/Qwen3.5-0.8B'
MODEL_REVISION = HfApi().model_info(BASE_MODEL).sha
FEATURE_DIM, MEMORY_RANK, CONTEXT, PROBE_CONTEXT, SEED = 32, 64, 128, 128, 73
MAX_ROUNDS_PER_RUN = 4
cfg = AutoConfig.from_pretrained(BASE_MODEL, revision=MODEL_REVISION).get_text_config(decoder=True)
full_layers = [i for i,k in enumerate(cfg.layer_types) if k == 'full_attention']
linear_layers = [i for i,k in enumerate(cfg.layer_types) if k == 'linear_attention']
print({'model':BASE_MODEL,'revision':MODEL_REVISION,'layers':cfg.num_hidden_layers,'full_attention_layers':full_layers,'native_linear_attention_layers':linear_layers,'hidden_size':cfg.hidden_size,'heads':cfg.num_attention_heads,'kv_heads':cfg.num_key_value_heads,'head_dim':cfg.head_dim,'max_context':cfg.max_position_embeddings})
assert full_layers == [3,7,11,15,19,23]


In [ ]:
env = dict(os.environ); env['TINYCENN_PARENT_BACKUP_ACTIVE'] = '1'
r = subprocess.run([sys.executable,'-m','pytest','-q','tests/test_qwen3_5_memory_fusion.py'], cwd=REPO, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(r.stdout)
if r.returncode: raise RuntimeError(f'preflight failed: {r.returncode}')
print('✅ Qwen3.5 preflight passed')


In [ ]:
cmd = [sys.executable,'-u',str(REPO/'scripts'/'train_qwen35_memory_fusion_sequential.py'),'--base-model',BASE_MODEL,'--model-revision',MODEL_REVISION,'--output-dir',str(OUTPUT_DIR),'--feature-dim',str(FEATURE_DIM),'--memory-rank',str(MEMORY_RANK),'--context-length',str(CONTEXT),'--probe-context',str(PROBE_CONTEXT),'--seed',str(SEED),'--min-layer-steps','50','--max-layer-steps','300','--check-every','25','--layer-lr','0.0002','--teacher-alpha-start','0.9','--teacher-alpha-end','0.0','--accept-nmse','0.2','--accept-cosine','0.9','--accept-incremental-delta-nll','0.015','--accept-cumulative-delta-nll','0.05','--max-runtime-minutes','240','--resume','--strict-acceptance']
print(' '.join(cmd), flush=True)
run_env = dict(os.environ); run_env['SEQUENTIAL_MAX_ROUNDS_PER_RUN'] = str(MAX_ROUNDS_PER_RUN)
result = subprocess.run(cmd, cwd=REPO, env=run_env)
if result.returncode: raise RuntimeError(f'trainer failed with exit code {result.returncode}')
print('✅ trainer returned normally; current_layer_needs_more_training is a resumable scientific status, not a crash')


In [ ]:
import json
for name in ('sequential_run_status.json','sequential_progress.json','sequential_in_progress.json','sequential_training_report.json'):
    p = OUTPUT_DIR/name
    if p.exists():
        print('\n###', name); print(p.read_text())


## Compare original Qwen3.5 with accepted Memory Fusion layers

Only accepted layers are loaded here. A current unaccepted layer is never used in the comparison.


In [ ]:
import torch, json
from transformers import AutoTokenizer, Qwen3_5ForCausalLM
from tinycenn_lm.qwen3_5_memory_fusion import Qwen35MemoryFusionConfig, replace_attention_layers, load_selected_attention_state, structural_summary
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE = torch.bfloat16 if DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported() else (torch.float16 if DEVICE.type == 'cuda' else torch.float32)
progress = OUTPUT_DIR/'sequential_progress.pt'
payload = torch.load(progress, map_location='cpu', weights_only=False) if progress.exists() else None
accepted = [int(x) for x in payload.get('accepted_layers',[])] if payload else []
mf_cfg = Qwen35MemoryFusionConfig.from_dict(payload['config']) if payload else Qwen35MemoryFusionConfig(feature_dim=FEATURE_DIM,memory_rank=MEMORY_RANK)
tok = AutoTokenizer.from_pretrained(BASE_MODEL, revision=MODEL_REVISION)
original = Qwen3_5ForCausalLM.from_pretrained(BASE_MODEL,revision=MODEL_REVISION,dtype=DTYPE,attn_implementation='sdpa',low_cpu_mem_usage=True).to(DEVICE).eval()
adapted = Qwen3_5ForCausalLM.from_pretrained(BASE_MODEL,revision=MODEL_REVISION,dtype=DTYPE,attn_implementation='sdpa',low_cpu_mem_usage=True).to(DEVICE).eval()
if accepted:
    replace_attention_layers(adapted,mf_cfg,accepted); load_selected_attention_state(adapted,payload['attention_state'],accepted)
adapted.config.use_cache = False
print('Accepted layers:', accepted); print(json.dumps(structural_summary(adapted), indent=2))
@torch.no_grad()
def gen(model,prompt):
    ids=tok(prompt,return_tensors='pt').input_ids.to(DEVICE)
    out=model.generate(ids,max_new_tokens=48,do_sample=False,use_cache=False,pad_token_id=tok.eos_token_id)
    return tok.decode(out[0],skip_special_tokens=True)
prompts=['The capital of Austria is','Explain in one sentence why the sky looks blue.','Write a short Python function that adds two numbers.','The largest planet in the Solar System is','2 + 2 =']
examples=[]
for i,prompt in enumerate(prompts,1):
    a,b=gen(original,prompt),gen(adapted,prompt); examples.append({'prompt':prompt,'original':a,'memory_fusion':b}); print('\n'+'='*100); print('PROMPT',i,prompt); print('\nORIGINAL QWEN3.5:\n',a); print('\nMEMORY FUSION (accepted only):\n',b)
(OUTPUT_DIR/'prompt_examples.json').write_text(json.dumps(examples,indent=2),encoding='utf-8')
print('✅ prompt smoke test complete')
